In [21]:
data = pd.read_csv('/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv')
volume = pd.read_csv('/Users/valter.rebelo/MissionControl/data/micro/assetData/bitcoin.csv')


,date,close,market_cap,total_volume
0,2013-04-27,135.300000,1.500518e+09,0.000000e+00
1,2013-04-28,141.960000,1.575032e+09,0.000000e+00
2,2013-04-29,135.300000,1.501657e+09,0.000000e+00
3,2013-04-30,117.000000,1.298952e+09,0.000000e+00
4,2013-05-01,103.430000,1.148668e+09,0.000000e+00
...,...,...,...,...
4323,2025-02-28,84441.901224,1.674754e+12,8.069524e+10
4324,2025-03-01,86005.256297,1.705564e+12,3.063447e+10
4325,2025-03-02,94261.532865,1.868322e+12,6.185911e+10
4326,2025-03-03,86124.714187,1.708199e+12,6.871536e+10


In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

# Load your BTC candle data
candle_data = pd.read_csv('/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv')
volume_data = pd.read_csv('/Users/valter.rebelo/MissionControl/data/micro/assetData/bitcoin.csv')

# Merge dataframes on 'date'
merged_data = pd.merge(candle_data, volume_data[['date', 'total_volume']], on='date', how='left')
merged_data.rename(columns={'total_volume': 'volume'}, inplace=True)
merged_data.dropna(inplace=True)
merged_data['date'] = pd.to_datetime(merged_data['date'])
merged_data.set_index('date', inplace=True)


merged_data['low_7'] = merged_data['low'].rolling(window=7).min()
merged_data['high_7'] = merged_data['high'].rolling(window=7).max()

merged_data['low_14'] = merged_data['low'].rolling(window=14).min()
merged_data['high_14'] = merged_data['high'].rolling(window=14).max()

merged_data['low_21'] = merged_data['low'].rolling(window=21).min()
merged_data['high_21'] = merged_data['high'].rolling(window=21).max()

merged_data['low_28'] = merged_data['low'].rolling(window=28).min()
merged_data['high_28'] = merged_data['high'].rolling(window=28).max()

merged_data['low_35'] = merged_data['low'].rolling(window=35).min()
merged_data['high_35'] = merged_data['high'].rolling(window=35).max()

merged_data['low_42'] = merged_data['low'].rolling(window=42).min()
merged_data['high_42'] = merged_data['high'].rolling(window=42).max()

merged_data['support_cluster'] = (merged_data['low_7']+merged_data['low_14']+merged_data['low_21']+merged_data['low_28']+merged_data['low_35']+merged_data['low_42'])/6
merged_data['resistance_cluster'] = (merged_data['high_7']+merged_data['high_14']+merged_data['high_21']+merged_data['high_28']+merged_data['high_35']+merged_data['high_42'])/6

# Calculate Bollinger Bands
# First, calculate the 20-day moving average
merged_data['ma20'] = merged_data['close'].rolling(window=20).mean()

# Calculate the standard deviation of the closing price over the same period
merged_data['std20'] = merged_data['close'].rolling(window=20).std()

# Calculate the upper and lower Bollinger Bands
# Upper band = 20-day MA + (20-day std * 2)
# Lower band = 20-day MA - (20-day std * 2)
merged_data['upper_band'] = merged_data['ma20'] + (merged_data['std20'] * 2)
merged_data['lower_band'] = merged_data['ma20'] - (merged_data['std20'] * 2)

# Calculate distance from bands
merged_data['distance_from_upper_band'] = merged_data['upper_band']/merged_data['close'] - 1
merged_data['distance_from_lower_band'] = merged_data['lower_band']/merged_data['close'] - 1

# Calculate rolling z-score normalization for distances (using 20-day window)
merged_data['distance_upper_band_zscore'] = (
    merged_data['distance_from_upper_band'] - 
    merged_data['distance_from_upper_band'].rolling(window=20).mean()
) / merged_data['distance_from_upper_band'].rolling(window=20).std()

merged_data['distance_lower_band_zscore'] = (
    merged_data['distance_from_lower_band'] - 
    merged_data['distance_from_lower_band'].rolling(window=20).mean()
) / merged_data['distance_from_lower_band'].rolling(window=20).std()

# Calculate log returns
merged_data['log_return'] = np.log(merged_data['close'] / merged_data['close'].shift(1))

# Calculate correlation between log returns and distances from bands
corr_upper_band = merged_data['log_return'].corr(merged_data['distance_from_upper_band'])
corr_lower_band = merged_data['log_return'].corr(merged_data['distance_from_lower_band'])

print(f"Correlation between log return and distance from upper band: {corr_upper_band:.4f}")
print(f"Correlation between log return and distance from lower band: {corr_lower_band:.4f}")



# Drop NaN values that result from the rolling calculations
merged_data.dropna(inplace=True)


Correlation between log return and distance from upper band: -0.3059
Correlation between log return and distance from lower band: -0.3166


In [37]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create a figure with subplots
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                   vertical_spacing=0.1, 
                   subplot_titles=('Support and Resistance Clusters', 'Bollinger Bands'))

# Add close price to first subplot
fig.add_trace(
    go.Scatter(
        x=merged_data.index,
        y=merged_data['close'],
        mode='lines',
        name='Close Price',
        line=dict(color='#1f77b4', width=1.5)
    ),
    row=1, col=1
)

# Add support cluster to first subplot
fig.add_trace(
    go.Scatter(
        x=merged_data.index,
        y=merged_data['support_cluster'],
        mode='lines',
        name='Support Cluster',
        line=dict(color='#2ca02c', width=1.5)
    ),
    row=1, col=1
)

# Add resistance cluster to first subplot
fig.add_trace(
    go.Scatter(
        x=merged_data.index,
        y=merged_data['resistance_cluster'],
        mode='lines',
        name='Resistance Cluster',
        line=dict(color='#d62728', width=1.5)
    ),
    row=1, col=1
)



# Add upper bollinger band to second subplot
fig.add_trace(
    go.Scatter(
        x=merged_data.index,
        y=merged_data['distance_upper_band_zscore'],
        mode='lines',
        name='Upper Band',
        line=dict(color='#ff7f0e', width=1.5)
    ),
    row=2, col=1
)


# Add lower bollinger band to second subplot
fig.add_trace(
    go.Scatter(
        x=merged_data.index,
        y=merged_data['distance_lower_band_zscore'],
        mode='lines',
        name='Lower Band',
        line=dict(color='#8c564b', width=1.5)
    ),
    row=2, col=1
)

# Update layout for a clean, elegant look
fig.update_layout(
    title='Bitcoin Price Analysis',
    template='plotly_white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(l=40, r=40, t=80, b=40),
    hovermode='x unified',
    height=800
)

# Update y-axis labels
fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
fig.update_yaxes(title_text="Price (USD)", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

# Show the figure
fig.show()
